# Ejercicio: comparar los modos de salida

Antes de ejecutar, abre una terminal y deja corriendo:

```
nc -lk 9999
```

Ejecuta las celdas en orden. No ejecutes `spark.stop()` hasta el final: después de eso hay que reiniciar el kernel.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import explode, split
import time

spark = SparkSession.builder \
    .appName("Streaming IABD WordCount") \
    .master("local[2]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [ ]:
lineasDF = spark.readStream \
    .format("socket") \
    .option("host", "localhost") \
    .option("port", "9999") \
    .load()

palabrasDF = lineasDF.select(explode(split(lineasDF.value, " ")).alias("palabra"))
cantidadDF = palabrasDF.groupBy("palabra").count()

## Funciones

Cada micro-batch se guarda en una lista para mostrarlo aquí mismo en el notebook.

In [ ]:
registro = []

def guardar(df, batch_id):
    filas = sorted((r["palabra"], r["count"]) for r in df.collect())
    registro.append((batch_id, filas))

def iniciar(modo):
    registro.clear()
    return cantidadDF.writeStream \
        .outputMode(modo) \
        .foreachBatch(guardar) \
        .start()

def mostrar():
    for batch_id, filas in list(registro):
        print(f"Batch {batch_id}: {filas}")

## Modo `complete`

1. Ejecuta la celda de abajo.
2. En `nc` envía `Esta es la primera frase` y luego `Enviamos segunda linea primera`.
3. Espera unos segundos y ejecuta `mostrar()`.
4. Detén la consulta.

In [ ]:
q = iniciar("complete")

In [ ]:
mostrar()

In [ ]:
q.stop()

## Modo `update`

Repite los mismos pasos con las mismas dos frases. Solo deben aparecer las filas que cambiaron en cada batch.

In [ ]:
q = iniciar("update")

In [ ]:
mostrar()

In [ ]:
q.stop()

## Modo `append`

Con una agregación sin watermark, Spark no lo permite y lanza una excepción.

In [ ]:
from pyspark.errors import AnalysisException

try:
    q = iniciar("append")
    q.stop()
except AnalysisException as e:
    print(str(e).splitlines()[0])

## Conclusión

- `complete`: en cada micro-batch muestra la tabla completa de conteos.
- `update`: muestra solo las palabras nuevas o cuyo conteo cambió.
- `append`: no es válido con agregaciones sin watermark, porque el conteo de una palabra puede cambiar con datos futuros.

## Cierre (ejecutar solo al terminar)

In [ ]:
spark.stop()